# 4. Agentic RAG: Function Calling

This notebook is the bridge from the fixed RAG pipeline to agentic systems. It follows the DataTalks.Club lessons on the Part 1 wrap-up, agents, RAG revision, and function calling.

## Learning objectives

By the end, you should be able to:

1. Explain why a fixed RAG pipeline cannot recover from a poor search result.
2. Describe an agent as an LLM that chooses actions through tools.
3. Define a tool with a JSON schema that the model can understand.
4. Parse a function call, execute the Python function, and return its result.
5. Preserve the conversation history and use `call_id` to link tool results.
6. Estimate the extra token and cost impact of tool-using requests.

This notebook demonstrates **one function-calling round trip**. The repeated `while` loop that supports multiple searches belongs in [05-agentic-loop-notes.md](05-agentic-loop-notes.md). Framework comparisons begin with [06-toyaikit-vs-handwritten-loop-notes.md](06-toyaikit-vs-handwritten-loop-notes.md).

## Why fixed RAG needs an agent

A fixed RAG pipeline follows a developer-defined sequence:

```text
user question -> one search -> build context -> one LLM answer
```

That works when the wording matches the index. With lexical search, a typo such as `Olama` can produce poor results. The LLM never sees the failed retrieval step, so it cannot correct the query and try again.

An agent changes who controls the next action:

```text
user question -> LLM decides -> tool call(s) -> tool results -> final answer
```

The model can rewrite a query, search more than once, ask for clarification, or decide that no tool is needed. This flexibility is the reason for the extra coordination and cost.

## Step 1: Initialize OpenAI Client

First, we load environment variables and create an OpenAI client. This client will handle all API calls to OpenAI's models, including our model that will have access to the search tool function.

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

## Step 2: Load FAQ Data and Build Search Index

We load course FAQ data from an external source and build a full-text search index using the `minsearch` library. This index will be used by the RAG system to quickly retrieve relevant Q&A pairs. The index is configured to search across question, section, and answer fields with boosted weights for question relevance.

In [ ]:
from ingestion import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

## Step 3: Create RAG Assistant

Instantiate the `RAGBase` class which encapsulates our RAG system. This assistant will combine the search index with the OpenAI LLM to answer questions by first retrieving relevant context from the FAQ database, then using that context to generate accurate responses.

In [ ]:
from rag_pipeline import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

## Step 4: Test Basic RAG Query

Let's test the RAG assistant with a straightforward question to see how it retrieves and uses FAQ data.

In [5]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: download and install the `.pkg`
     - **Windows**: download and install the `.msi`
     - **Linux**: run:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a local model**
   Open a terminal and run:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. **Check the local server**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response like:
   ```json
   {"models": [...]}  
   ```

4. **If you get a connection refused error**
   Restart the Ollama server with:
   ```bash
   !nohup ollama serve > nohup.out 2>&1 &
   ```

If you want, I can also show the minimal Python example for using Ollama locally.


## Step 5: Test a deliberate misspelling

Deliberately misspell `Ollama` as `Olama` to expose the limitation of a fixed lexical pipeline.

The search runs once using the exact user query. If the index returns poor results, the LLM receives poor context and has no built-in way to rewrite the query or retry.

**Expected learning:** this failure is the motivation for function calling and the repeated agent loop.

In [6]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I don’t see anything in the context about **Olama** specifically.

If you mean running the **course locally**, the FAQ says you can do that instead of using Codespaces, but you need to be comfortable setting up:

- Python
- `uv`
- Jupyter
- Docker
- any other tools needed for the module

If you run locally, you should also **document your setup** and keep it **reproducible**.

If you meant something else by “Olama,” let me know.


## Step 6: Define the Python search tool

Before exposing a tool to the model, define the ordinary Python function that performs the work. This keeps the application-owned behavior separate from the model-facing schema.

The model will reference this function by name, so keeping the Python name and tool name aligned makes dispatch easier later.

```python
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )
```

The function takes a query, applies the retrieval policy, and returns the top five FAQ entries. The model does not see this implementation; it sees the schema in the next step.

In [7]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Yes—probably, but it depends on the course’s enrollment status and whether there’s still room.\n\nIf you want, I can help you figure it out by checking:\n- the course start date\n- whether enrollment is still open\n- any prerequisites\n- how to register\n\nIf you already have the course name or link, send it and I’ll help you next.'

## Step 7: Ask without tools

Before adding function calling, establish the baseline. Ask the same course question without giving the model access to the FAQ search tool.

Without a tool, the model can only use its general knowledge and may give generic advice. It cannot inspect course-specific policies, deadlines, or procedures.

**Interactive checkpoint:** compare this answer with the answer after the tool result is returned. What became more specific once the model received the FAQ evidence?

In [8]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

## Step 8: Describe the search tool with JSON Schema

The model cannot inspect the Python function directly. It receives a schema containing the tool name, purpose, and valid arguments.

The `description` is especially important: the model uses it to decide when the tool is relevant. The `parameters` object constrains the arguments it can produce, and `required` prevents an incomplete call.

**Interactive checkpoint:** rewrite the description so it clearly says when the tool should be used and what kind of query it expects. A vague description can lead to skipped tools or poor arguments.

The schema is language-independent. A Python, TypeScript, or Java application can expose the same tool contract to the model.

## One complete function-calling round trip

The model first returns a `function_call` instead of a final answer. Our application then:

1. parses the JSON arguments;
2. dispatches to the matching Python function;
3. serializes the result as JSON;
4. appends the model's tool request and the tool output to `messages`;
5. calls the model again with the complete history.

The `call_id` is the correlation key between a requested function call and its result. This becomes essential when a response contains multiple tool calls.

The model is stateless between API requests. The `messages` list is the memory we send back on the second call, including the original question, the model's decision, and the retrieved evidence.

This is agentic RAG at the single-turn level. For an unknown number of searches, add the repeated control flow described in [05-agentic-loop-notes.md](05-agentic-loop-notes.md).

In [9]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

## Step 9: Send the question with the tool available

Send the same question again, this time including `search_tool` in the request. The model can now choose between answering directly and requesting retrieval.

In [10]:
len(response.output)

1

## Step 10: Inspect the model's response

Count the returned items and inspect `response.output`. The expected result is a `function_call`, not a final answer. The model has decided that it needs the search tool first.

In [ ]:
# Step 11: Extract the function call.
# Inspect the function call that the model generated.
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enroll registration late join eligibility"}', call_id='call_4T5chAJC2cbywi1lHpgLaW4v', name='search', type='function_call', id='fc_09a3f4ede7c53d63006a7470ee8c0c8191b027cbba671fecfd', namespace=None, status='completed')]

In [12]:
call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enroll registration late join eligibility"}', call_id='call_4T5chAJC2cbywi1lHpgLaW4v', name='search', type='function_call', id='fc_09a3f4ede7c53d63006a7470ee8c0c8191b027cbba671fecfd', namespace=None, status='completed')

In [ ]:
# Step 12: Parse the function arguments.
import json

args = json.loads(call.arguments)
args

NameError: name 'args' is not defined

In [14]:
import json

args = json.loads(call.arguments)
args

{'query': 'join course discovered course can I join enroll registration late join eligibility'}

## Step 11: Verify the requested function

In [15]:
call.name

'search'

## Step 12: Execute the search function

In [16]:
results = assistant.search(**args)

## Step 13: Serialize the search results

In [17]:
result_json = json.dumps(results, indent=2)

## Step 14: Create the function output message

In [18]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

## Step 15: Add the tool call to the history

In [19]:
messages.append(call)

## Step 16: Add the function output to the history

In [20]:
messages.append(function_call_output)

## Step 17: Inspect the complete conversation

In [21]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enroll registration late join eligibility"}', call_id='call_4T5chAJC2cbywi1lHpgLaW4v', name='search', type='function_call', id='fc_09a3f4ede7c53d63006a7470ee8c0c8191b027cbba671fecfd', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_4T5chAJC2cbywi1lHpgLaW4v',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can 

## Step 18: Make the final API call with context

In [22]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

## Step 19: Display the grounded answer

In [23]:
print(response.output_text)

Yes — you can still join and start learning.

If you want a certificate, though, you need to submit your project while the course is still accepting submissions.


## Step 20: Monitor token usage

In [24]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(774, 36)

## Step 21: Token usage, cost, and trade-offs

Tool use usually requires at least two model calls: one to decide whether and how to call the tool, and another to interpret the result and answer. Every call has its own input and output usage.

Remember to add the usage from **all** calls in a real workflow. The second call can be especially expensive because it resends the conversation history, tool schema, and function result.

A useful development checklist is:

- log input and output tokens per call;
- track total cost per user request;
- record the number of tool calls and iterations;
- set limits for maximum iterations, tokens, and wall-clock time;
- compare the quality improvement against the extra latency and cost.

**Interactive checkpoint:** compare the plain RAG request from notebook 02 with this two-call flow. Which extra information improves the answer, and which tokens are repeated?

In [25]:
def calculate_gpt54mini_price(usage):
    """Calculate cost for gpt-5.4-mini API usage."""
    input_cost_per_1M = 0.15   # $0.15 per 1M input tokens
    output_cost_per_1M = 0.60  # $0.60 per 1M output tokens
    
    input_cost = (usage.input_tokens / 1_000_000) * input_cost_per_1M
    output_cost = (usage.output_tokens / 1_000_000) * output_cost_per_1M
    
    total_cost = input_cost + output_cost
    
    return {
        'input_tokens': usage.input_tokens,
        'output_tokens': usage.output_tokens,
        'input_cost': round(input_cost, 6),
        'output_cost': round(output_cost, 6),
        'total_cost': round(total_cost, 6)
    }

# Calculate cost for this interaction
cost_breakdown = calculate_gpt54mini_price(usage)
print(f"Input tokens: {cost_breakdown['input_tokens']}")
print(f"Output tokens: {cost_breakdown['output_tokens']}")
print(f"Input cost: ${cost_breakdown['input_cost']}")
print(f"Output cost: ${cost_breakdown['output_cost']}")
print(f"Total cost: ${cost_breakdown['total_cost']}")

Input tokens: 774
Output tokens: 36
Input cost: $0.000116
Output cost: $2.2e-05
Total cost: $0.000138


# What comes next

You now have the core contract: instructions, tools, and conversation history. The next learning notes extend this foundation:

- [05-agentic-loop-notes.md](05-agentic-loop-notes.md) adds the `while` loop, multiple searches, guardrails, and a reusable `agent_loop()` function.
- [06-toyaikit-vs-handwritten-loop-notes.md](06-toyaikit-vs-handwritten-loop-notes.md) compares the handwritten loop with ToyAIKit.
- [07-pydantic-ai-vs-toyaikit-notes.md](07-pydantic-ai-vs-toyaikit-notes.md) adds PydanticAI's typed tools and dependency injection.

## Two directions after the RAG foundation

The Part 1 RAG system can improve in two different directions:

- **Better retrieval**: keyword search is lexical, so semantic or vector search can handle different wording. A production search backend such as Elasticsearch or OpenSearch adds BM25, filtering, scaling, and vector-search options.
- **More flexible control**: an agent lets the model decide whether to search, how to rewrite a query, whether to search again, and when to stop.

Fine-tuning is a different trade-off. It changes model weights and may help with consistent style or behavior, but it is harder to update when the knowledge changes. RAG keeps changing information in the retrieval layer and works with many models, so it is usually the first approach to try for external or frequently updated knowledge.

**Decision checkpoint:** use plain RAG when the workflow is predictable and one retrieval step is enough. Use an agent when the system genuinely needs query rewriting, multiple tools, clarification, or dynamic control flow.

## Adapting This Pattern to Other Use Cases

The function calling pattern you learned here is universal. Here's how to adapt it:

### Example: Power BI Report Search

Instead of searching FAQs, imagine searching Power BI reports:

```python
def search_powerbi_reports(query, workspace=None):
    """Search Power BI reports by name or description."""
    # Your Power BI API logic here
    return matching_reports

# Tool schema
powerbi_tool = {
    "type": "function",
    "function": {
        "name": "search_powerbi_reports",
        "description": "Search for Power BI reports by name or description. Use this when the user asks about reports, dashboards, or analytics.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query for report names or descriptions"
                },
                "workspace": {
                    "type": "string",
                    "description": "Optional: filter by workspace name"
                }
            },
            "required": ["query"],
            "additionalProperties": false
        }
    }
}
```

### Key Adaptation Points

1. **Function Implementation**: Change what the function does (API call, database query, calculation)
2. **Tool Schema**: Update `name`, `description`, and `parameters` to match your function
3. **Function Execution**: Parse `call.arguments` and call your function with `**args`
4. **Everything Else Stays the Same**: Message history, `call_id` linking, multi-turn pattern

### Multi-Tool Systems

You can provide multiple tools at once:

```python
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool, powerbi_tool, calculator_tool]
)
```

The model will:
- Choose the right tool for the job
- Or call multiple tools in sequence
- Or decide no tool is needed

This makes the system truly agentic!